# 03 — Analytics Library Demo

> **SYNTHETIC DATA DEMO** — All telemetry and fault scenarios in this notebook are programmatically generated. Results demonstrate architecture and controlled functional behaviour, not real-world accuracy or statistical validation.

Exercises the deterministic analytics independently of the LLM. Ground truth is not needed for these calculations.

In [1]:
# Colab/local bootstrap: discover the repository, clone only when needed.
from pathlib import Path
import os, sys, subprocess

start = Path.cwd().resolve()
candidates = [start, *start.parents]
ROOT = next((p for p in candidates if (p / "src").exists() and (p / "config").exists()), None)
if ROOT is None:
    if Path("/content").exists():
        repo = Path("/content/intelligent-data-logger-demo")
        if not repo.exists():
            subprocess.run(["git", "clone", "https://github.com/Engr-Daniel/intelligent-data-logger-demo.git", str(repo)], check=True)
        ROOT = repo
    else:
        raise RuntimeError("Could not locate the repository. Open the notebook from the cloned repo or use Colab.")
os.chdir(ROOT)
if Path("/content").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository root:", ROOT)

Repository root: /mnt/data/m5work/intelligent-data-logger-demo


In [2]:
from src.reasoning.tools import execute_tool

checks = [
    ("Weather drop", "investigate_generation_drop", {"question":"Why did generation drop?","target_date":"2026-08-05"}),
    ("Overload", "investigate_inverter_failure", {"question":"Why did the inverter fail?"}),
    ("PV trend", "investigate_performance_trend", {"question":"Has performance declined?"}),
    ("Energy", "get_energy_summary", {"question":"Solar contribution?","days":30}),
    ("Runway", "get_battery_runway", {"question":"Battery runway?"}),
    ("Sustainability", "get_sustainability_summary", {"question":"Sustainability?","days":30}),
    ("Financial", "get_financial_summary", {"question":"ROI?"}),
    ("Data quality", "assess_data_quality", {"question":"Enough data?","target_date":"2026-09-18"}),
]
for label, tool, args in checks:
    e = execute_tool(tool, args)
    print(f"\n{label}: {e['finding']}")
    print("  candidate:", e.get("candidate_cause"), "confidence:", e.get("confidence"))
    print("  tools:", e.get("supporting_tools"))


Weather drop: PV production was materially lower and the drop closely tracked lower irradiance, with no inverter alarm or abnormal operating state.
  candidate: weather_related_low_irradiance confidence: high
  tools: ['analyze_generation_drop']



Overload: Sustained overload preceded the inverter alarm.
  candidate: overload confidence: high
  tools: ['diagnose_overload']



PV trend: Observed irradiance- and temperature-normalized PV performance shows a sustained downward multi-week trend.
  candidate: gradual_pv_performance_decline confidence: high
  tools: ['detect_gradual_performance_decline']



Energy: Energy balance for the latest 30 days was calculated from stored telemetry.
  candidate: None confidence: high
  tools: ['energy_balance']



Runway: Battery runtime was estimated at the latest complete observation using a constant-load assumption.
  candidate: None confidence: medium
  tools: ['battery_runway']



Sustainability: Sustainability metrics for the latest 30 days were calculated using the configured grid-displacement assumption.
  candidate: None confidence: medium
  tools: ['sustainability_metrics']



Financial: Simple financial metrics were calculated from synthetic telemetry and configured tariff/CAPEX assumptions.
  candidate: None confidence: medium
  tools: ['financial_metrics']



Data quality: Telemetry is unavailable for part of 2026-09-18; a causal diagnosis should abstain for the affected interval.
  candidate: telemetry_unavailable confidence: high
  tools: ['assess_data_availability']


Known limitation retained intentionally: the reviewer-approved M3/M4 baseline does **not** yet include an observed-telemetry battery-capacity degradation diagnostic. M5 scores that gap rather than hiding it.